In [5]:
from docplex.mp.model import Model
import networkx as nx
import numpy as np

def maxcut_mdl(graph):
    mdl = Model(name='maxcut')
    nodes = list(graph.nodes)
    x = mdl.binary_var_dict(nodes, name='x')
    mdl.maximize(mdl.sum(graph[i][j]['weight'] * (x[i] + x[j] - 2 * x[i] * x[j]) for i, j in graph.edges))
    return mdl

nq = 10
graph = nx.Graph()

graph.add_nodes_from(range(nq))
for i in range(nq):
    for j in range(i+1, nq):
        graph.add_edge(i, j, weight=np.random.rand())

mdl = maxcut_mdl(graph)
solution = mdl.solve()
print("Optimal value:", solution.objective_value)
print("Solution:")
for v in mdl.iter_variables():
    print(f"{v.name}: {v.solution_value}")
    optimal_sol = "".join(str(int(v.solution_value)) for v in mdl.iter_variables())

Optimal value: 14.383640017428492
Solution:
x_0: 0
x_1: 1.0
x_2: 1.0
x_3: 1.0
x_4: 1.0
x_5: 0
x_6: 0
x_7: 1.0
x_8: 1.0
x_9: 0


In [ ]:
def ising_dict(graph):
    n = len(graph.nodes)
    h = {}
    J = {}
    for i, j in graph.edges:
        weight = graph[i][j]['weight']
        J[i,j] = -weight / 2
    return h, J

In [16]:
from dimod import BinaryQuadraticModel
from dimod import SimulatedAnnealingSampler

sa_sampler = SimulatedAnnealingSampler()

bqm = BinaryQuadraticModel.from_ising(*ising_dict(graph))

In [17]:
sa_sampleset = sa_sampler.sample(bqm, num_reads=100)